# Test: `ClassificationPicker`

This notebook imports `classification_picker_1a.py` directly and adds the built-in charts. Each part ends with chart checks that call every `plot_` method.

This notebook uses two well-known datasets:

| Part | Dataset | Why it's a good test |
|---|---|---|
| **A** | **IBM Telco Customer Churn** (~7,000 customers) | Binary target (`Churn` Yes/No). Classes are uneven (about 27% churn), there are many text columns, and `TotalCharges` has hidden blanks to clean |
| **B** | **Palmer Penguins** (~340 penguins) | Three classes (species), missing values, and text columns (`island`, `sex`). Ranked by AUC |

Each part ends with automatic checks that print **All checks passed**.

In [ ]:
# --- Find the ML root folder and import the picker ---
import sys
from pathlib import Path

ml_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pickers").is_dir())
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pickers.classification_picker_1a import ClassificationPicker
print("Using:", ml_root / "pickers" / "classification_picker_1a.py")

# Part A: Telco Customer Churn (binary)
## A1. Load the data

In [ ]:
CHURN_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
churn_raw = pd.read_csv(CHURN_URL)
print(churn_raw.shape)
churn_raw.head()

## A2. Clean it
- `customerID` is just an identifier, so we drop it.
- `TotalCharges` *looks* numeric but was read as text, because 11 brand-new customers have a blank `" "` instead of a number.
  `isna()` won't find those, which makes it a good reminder to check dtypes, not just missing values.

In [ ]:
print("TotalCharges dtype:", churn_raw["TotalCharges"].dtype)

churn = churn_raw.drop(columns="customerID")
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")   # blanks -> NaN
print("Missing after conversion:", churn["TotalCharges"].isna().sum())

churn_unclean = churn.copy()            # kept for the 'rejects missing values' check
churn = churn.dropna().reset_index(drop=True)
print("Rows after cleaning:", len(churn))
churn["Churn"].value_counts(normalize=True).round(3)

## A3. Set up the picker
- `positive_class="Yes"`: Recall, Precision and F1 are measured on catching churners.
- `sort="F1"`: with uneven classes, accuracy is misleading. A model that predicts "No" for everyone is already about 73% accurate.
- The classes are uneven, so `class_weight="balanced"` should be switched on automatically as the starting value.

In [ ]:
cp = ClassificationPicker(churn, target="Churn", positive_class="Yes", sort="F1")

print(cp.summary())
print()
for m in ["lr", "knn", "dt", "rf", "hgb"]:
    print(f"{m:5}", cp.starting_params(m))

## A4. Compare all models (stratified 5-fold CV)

In [ ]:
%%time
churn_lb = cp.compare_models()
churn_lb

In [ ]:
cp.plot_leaderboard();

Look at the **Dummy** row: high accuracy, but an F1 of 0, because it never predicts a churner.
That's the accuracy trap with uneven classes.

In [ ]:
# The accuracy trap in one chart: the dummy looks fine on accuracy
cp.plot_leaderboard(metric="Accuracy");

## A5. Tune the top 3, then check on the holdout set

In [ ]:
%%time
churn_tuned = cp.tune_top(n=3, n_iter=20)
churn_tuned

In [ ]:
cp.plot_tuning();

In [ ]:
print(cp.summary())

In [ ]:
best_churn = cp.tuned_.index[0]
print(cp.class_report(best_churn))
cp.confusion_matrix(best_churn)

## A6. Charts for the best model (holdout set)

In [ ]:
cp.plot_confusion_matrix();

In [ ]:
# Row percentages: what share of real churners did we catch?
cp.plot_confusion_matrix(normalize=True);

In [ ]:
cp.plot_roc();                     # the 3 tuned models

In [ ]:
cp.plot_precision_recall();        # more honest than ROC when the positive class is rare

The threshold chart shows the trade-off behind every prediction. The best F1 is often **not** at the default threshold of 0.5.

In [ ]:
cp.plot_threshold();

In [ ]:
cp.plot_feature_importance();

## A7. Use the final model
`finalize()` refits on all rows, and its predictions come back as the original labels (`'Yes'` / `'No'`).

In [ ]:
churn_model = cp.finalize()

sample = churn.drop(columns="Churn").head(8)
proba_yes = churn_model.predict_proba(sample)[:, list(churn_model.classes_).index("Yes")]
pd.DataFrame({"actual": churn["Churn"].head(8).values,
              "predicted": churn_model.predict(sample),
              "P(churn)": proba_yes.round(3)})

## A8. Automatic checks: churn

In [ ]:
lb, tu = cp.leaderboard_, cp.tuned_
best = tu.index[0]

checks = {
    "leaderboard has 15+ models":                       len(lb) >= 15,
    "class_weight='balanced' switched on automatically": cp.ctx.balanced,
    "positive class is 'Yes'":                          cp.classes_[1] == "Yes",
    "accuracy trap: dummy accuracy > 0.70 but F1 = 0":  lb.loc["dummy", "Accuracy"] > 0.70 and lb.loc["dummy", "F1"] == 0,
    "best CV F1 beats dummy by 0.5+":                   lb["F1"].max() > lb.loc["dummy", "F1"] + 0.5,
    "15 text columns were one-hot encoded":             len(cp.cat_cols) == 15,
    "3 models were tuned, each with params":            len(tu) == 3 and all(m in cp.recommended_params_ for m in tu.index),
    "best holdout AUC is above 0.80":                   tu.loc[best, "Holdout AUC"] > 0.80,
    "confusion matrix covers the whole holdout set":    cp.confusion_matrix(best).values.sum() == len(cp.y_test),
    "final model predicts 'Yes'/'No' labels":           set(churn_model.predict(sample)) <= {"Yes", "No"},
    "missing values are rejected":                      False,
}
try:
    ClassificationPicker(churn_unclean, target="Churn")
except ValueError:
    checks["missing values are rejected"] = True

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some churn checks failed - see above"
print("\nAll churn checks passed")

## A9. Chart checks: churn

In [ ]:
from matplotlib.axes import Axes
from matplotlib.figure import Figure

def run_chart_checks(calls, expect_errors=None):
    """Call every chart; PASS if it returns a matplotlib Axes/Figure without an error.
    expect_errors: {name: (callable, ExceptionType)} that SHOULD raise that error."""
    results = {}
    for name, call in calls.items():
        try:
            results[name] = isinstance(call(), (Axes, Figure))
        except Exception as e:
            print(f"  {name} -> {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, (call, err) in (expect_errors or {}).items():
        try:
            call()
            results[name] = False
        except err:
            results[name] = True
        except Exception as e:
            print(f"  {name} -> unexpected {type(e).__name__}: {e}")
            results[name] = False
        finally:
            plt.close("all")
    for name, ok in results.items():
        print(("PASS  " if ok else "FAIL  ") + name)
    return results


In [ ]:
calls = {
    "plot_leaderboard()":                          lambda: cp.plot_leaderboard(),
    "plot_leaderboard(metric='AUC', top=5)":       lambda: cp.plot_leaderboard(metric="AUC", top=5),
    "plot_tuning()":                               lambda: cp.plot_tuning(),
    "plot_confusion_matrix()":                     lambda: cp.plot_confusion_matrix(),
    "plot_confusion_matrix(normalize=True)":       lambda: cp.plot_confusion_matrix(normalize=True),
    "plot_roc() - tuned models":                   lambda: cp.plot_roc(),
    "plot_roc() - decision-score models":          lambda: cp.plot_roc(["svm", "ridge"]),
    "plot_precision_recall()":                     lambda: cp.plot_precision_recall(),
    "plot_threshold()":                            lambda: cp.plot_threshold(),
    "plot_threshold() - decision-score model":     lambda: cp.plot_threshold("svm"),
    "plot_feature_importance(top=5)":              lambda: cp.plot_feature_importance(top=5, n_repeats=2),
}
expect = {"bad metric name raises ValueError":    (lambda: cp.plot_leaderboard(metric="RMSE"), ValueError)}
chart_results = run_chart_checks(calls, expect)
assert all(chart_results.values()), "Some churn chart checks failed - see above"
print("\nAll churn chart checks passed")

# Part B: Palmer Penguins (three classes, ranked by AUC)
## B1. Load and clean

In [ ]:
PENGUIN_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins_raw = pd.read_csv(PENGUIN_URL)
print("Missing values:\n", penguins_raw.isna().sum()[penguins_raw.isna().sum() > 0])

penguins = penguins_raw.dropna().reset_index(drop=True)
print("\nRows after cleaning:", len(penguins))
penguins["species"].value_counts()

## B2. Compare and tune
With `sort="AUC"`, models that can't give probability scores (Ridge Classifier, Linear SVM) are skipped with a note.
With three classes, AUC is averaged one-vs-rest and weighted by class size.

In [ ]:
%%time
pp = ClassificationPicker(penguins, target="species", sort="AUC")
peng_lb = pp.compare_models()
peng_lb

In [ ]:
peng_tuned = pp.tune_top(n=3, n_iter=15)
print(pp.summary())
print()
print(pp.class_report(pp.tuned_.index[0]))
pp.confusion_matrix(pp.tuned_.index[0])

## B3. Charts with three classes
With more than two classes, `plot_roc()` and `plot_precision_recall()` draw one curve per class (each class vs the rest) for a single model.

In [ ]:
pp.plot_leaderboard();

In [ ]:
pp.plot_confusion_matrix();

In [ ]:
pp.plot_roc();

## B4. Automatic checks: penguins

In [ ]:
lb, tu = pp.leaderboard_, pp.tuned_
best = tu.index[0]
species = {"Adelie", "Chinstrap", "Gentoo"}
peng_model = pp.finalize()

checks = {
    "3 classes found":                              pp.ctx.k == 3 and set(pp.classes_) == species,
    "AUC sort skipped Ridge and Linear SVM":        {"ridge", "svm"} <= set(pp.skipped_),
    "island and sex were one-hot encoded":          set(pp.cat_cols) == {"island", "sex"},
    "best holdout accuracy is above 0.95":          tu.loc[best, "Holdout Accuracy"] > 0.95,
    "confusion matrix is 3 x 3":                    pp.confusion_matrix(best).shape == (3, 3),
    "final model predicts species names":           set(peng_model.predict(penguins.drop(columns="species"))) <= species,
    "probabilities sum to 1":                       np.allclose(peng_model.predict_proba(penguins.drop(columns="species")).sum(axis=1), 1),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some penguin checks failed - see above"
print("\nAll checks passed")

## B5. Chart checks: penguins

In [ ]:
calls = {
    "plot_leaderboard()":                       lambda: pp.plot_leaderboard(),
    "plot_tuning()":                            lambda: pp.plot_tuning(),
    "plot_confusion_matrix(normalize=True)":    lambda: pp.plot_confusion_matrix(normalize=True),
    "plot_roc() - one curve per class":         lambda: pp.plot_roc(),
    "plot_precision_recall('lr')":              lambda: pp.plot_precision_recall("lr"),
    "plot_feature_importance()":                lambda: pp.plot_feature_importance(n_repeats=2),
}
expect = {"plot_threshold() on 3 classes raises ValueError": (lambda: pp.plot_threshold(), ValueError)}
chart_results = run_chart_checks(calls, expect)
assert all(chart_results.values()), "Some penguin chart checks failed - see above"
print("\nAll checks passed")